In [1]:
#Ridge

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge

X = pd.read_csv("../data/processed/X_features.csv", index_col=0)
Y = pd.read_csv("../data/processed/response_delta.csv", index_col=0)
groups = pd.read_csv("../data/processed/row_groups.csv", index_col=0)["group"]
meta = pd.read_csv("../data/processed/response_meta.csv", index_col=0)

Xv, Yv, gv = X.values, Y.values, groups.values

In [5]:
P_ridge = np.zeros_like(Yv)
best_alpha = []

gkf = GroupKFold(n_splits=5)
for tr, te in gkf.split(Xv, Yv, gv):
    search = GridSearchCV(
        Ridge(),
        {"alpha": [1000, 3000, 10000, 30000, 100000]},
        cv=GroupKFold(n_splits=4),        # inner split, training drugs only
    )
    search.fit(Xv[tr], Yv[tr], groups=gv[tr])
    P_ridge[te] = search.predict(Xv[te])
    best_alpha.append(search.best_params_["alpha"])

print("best alpha per fold:", best_alpha)

best alpha per fold: [1000, 3000, 1000, 1000, 1000]


In [7]:
def row_r(P, T):
    return np.array([np.corrcoef(p, t)[0, 1] for p, t in zip(P, T)])

scores = pd.read_csv("../results/tables/05_baseline_scores.csv", index_col=0)
scores["r_ridge"] = row_r(P_ridge, Yv)
scores["mse_ridge"] = ((P_ridge - Yv) ** 2).mean(axis=1)

cols = ["r_mean", "r_knn", "r_ridge", "mse_mean", "mse_knn", "mse_ridge"]
print("All rows:")
print(scores[cols].mean().round(4))
print("Responders only:")
print(scores.loc[scores["responder"], cols].mean().round(4))

All rows:
r_mean       0.2508
r_knn        0.1595
r_ridge      0.2642
mse_mean     0.0038
mse_knn      0.0063
mse_ridge    0.0036
dtype: float64
Responders only:
r_mean       0.4229
r_knn        0.3334
r_ridge      0.4985
mse_mean     0.0105
mse_knn      0.0120
mse_ridge    0.0099
dtype: float64


In [4]:
scores.to_csv("../results/tables/06_model_scores.csv")
np.save("../results/models/P_ridge.npy", P_ridge)

In [8]:
resp = scores[scores["responder"]]
wins = (resp["r_ridge"] > resp["r_mean"]).mean()
print("Ridge better than mean on", round(wins * 100), "% of responders")

Ridge better than mean on 73 % of responders


In [9]:
#Random Forest
from sklearn.decomposition import PCA

pca = PCA(n_components=50).fit(Yv)
print("variance kept by 50 PCs:", round(pca.explained_variance_ratio_.sum(), 3))

variance kept by 50 PCs: 0.833


In [10]:
from sklearn.ensemble import RandomForestRegressor

P_rf = np.zeros_like(Yv)

for tr, te in gkf.split(Xv, Yv, gv):
    pca = PCA(n_components=50).fit(Yv[tr])          # training rows only
    Z_tr = pca.transform(Yv[tr])                    # 2000 genes -> 50 patterns

    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                               n_jobs=-1, random_state=42)
    rf.fit(Xv[tr], Z_tr)

    P_rf[te] = pca.inverse_transform(rf.predict(Xv[te]))   # 50 -> 2000 genes

In [11]:
scores["r_rf"] = row_r(P_rf, Yv)
scores["mse_rf"] = ((P_rf - Yv) ** 2).mean(axis=1)

cols = ["r_mean", "r_ridge", "r_rf", "mse_mean", "mse_ridge", "mse_rf"]
print("All rows:")
print(scores[cols].mean().round(4))
print("Responders only:")
print(scores.loc[scores["responder"], cols].mean().round(4))

All rows:
r_mean       0.2508
r_ridge      0.2642
r_rf         0.2963
mse_mean     0.0038
mse_ridge    0.0036
mse_rf       0.0030
dtype: float64
Responders only:
r_mean       0.4229
r_ridge      0.4985
r_rf         0.5439
mse_mean     0.0105
mse_ridge    0.0099
mse_rf       0.0074
dtype: float64


In [12]:
scores.to_csv("../results/tables/06_model_scores.csv")
np.save("../results/models/P_rf.npy", P_rf)

In [13]:
P_ridge_pca = np.zeros_like(Yv)

for tr, te in gkf.split(Xv, Yv, gv):
    pca = PCA(n_components=50).fit(Yv[tr])
    Z_tr = pca.transform(Yv[tr])
    model = Ridge(alpha=1000).fit(Xv[tr], Z_tr)
    P_ridge_pca[te] = pca.inverse_transform(model.predict(Xv[te]))

scores["r_ridge_pca"] = row_r(P_ridge_pca, Yv)
resp = scores[scores["responder"]]
print("Responders r:", resp[["r_ridge", "r_ridge_pca", "r_rf"]].mean().round(4))
print("RF better than Ridge on", round((resp["r_rf"] > resp["r_ridge"]).mean() * 100), "% of responders")

Responders r: r_ridge        0.4985
r_ridge_pca    0.5019
r_rf           0.5439
dtype: float64
RF better than Ridge on 70 % of responders


In [16]:
#XG boost
from xgboost import XGBRegressor

P_xgb = np.zeros_like(Yv)

for tr, te in gkf.split(Xv, Yv, gv):
    pca = PCA(n_components=50).fit(Yv[tr])
    Z_tr = pca.transform(Yv[tr])

    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.5,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    xgb.fit(Xv[tr], Z_tr)

    P_xgb[te] = pca.inverse_transform(xgb.predict(Xv[te]))


In [17]:
scores["r_xgb"] = row_r(P_xgb, Yv)
scores["mse_xgb"] = ((P_xgb - Yv) ** 2).mean(axis=1)

cols = ["r_mean", "r_ridge", "r_rf", "r_xgb", "mse_mean", "mse_ridge", "mse_rf", "mse_xgb"]
print("All rows:")
print(scores[cols].mean().round(4))
print("Responders only:")
print(scores.loc[scores["responder"], cols].mean().round(4))

resp = scores[scores["responder"]]
print("XGB better than RF on", round((resp["r_xgb"] > resp["r_rf"]).mean() * 100), "% of responders")

All rows:
r_mean       0.2508
r_ridge      0.2642
r_rf         0.2963
r_xgb        0.2511
mse_mean     0.0038
mse_ridge    0.0036
mse_rf       0.0030
mse_xgb      0.0031
dtype: float64
Responders only:
r_mean       0.4229
r_ridge      0.4985
r_rf         0.5439
r_xgb        0.4825
mse_mean     0.0105
mse_ridge    0.0099
mse_rf       0.0074
mse_xgb      0.0073
dtype: float64
XGB better than RF on 33 % of responders


In [18]:
scores.to_csv("../results/tables/06_model_scores.csv")
np.save("../results/models/P_xgb.npy", P_xgb)

In [22]:
#MLP
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

P_mlp2 = np.zeros_like(Yv)

dose_col = list(X.columns).index("log_dose")

for tr, te in gkf.split(Xv, Yv, gv):
    pca = PCA(n_components=50).fit(Yv[tr])
    Z_tr = pca.transform(Yv[tr])
    s = Z_tr[:, 0].std()              # one scale number for all patterns
    Z_tr = Z_tr / s

    X_tr = Xv[tr].astype(float).copy()
    X_te = Xv[te].astype(float).copy()
    X_tr[:, dose_col] = (X_tr[:, dose_col] - 1) / 3   # dose 1..4 -> 0..1
    X_te[:, dose_col] = (X_te[:, dose_col] - 1) / 3

    mlp = MLPRegressor(
        hidden_layer_sizes=(256,),
        alpha=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        max_iter=500,
        random_state=42,
    )
    mlp.fit(X_tr, Z_tr)

    P_mlp2[te] = pca.inverse_transform(mlp.predict(X_te) * s)

In [25]:
scores["r_mlp2"] = row_r(P_mlp2, Yv)
scores["mse_mlp2"] = ((P_mlp2 - Yv) ** 2).mean(axis=1)

resp = scores[scores["responder"]]
print(resp[["r_mean", "r_rf", "r_mlp", "r_mlp2"]].mean().round(4))
print(resp[["mse_mean", "mse_rf", "mse_mlp", "mse_mlp2"]].mean().round(4))
print("Responders only:")
print(scores.loc[scores["responder"], cols].mean().round(4))
print("Responders MSE:")
print(scores.loc[scores["responder"], ["mse_mean", "mse_rf", "mse_mlp"]].mean().round(4))

resp = scores[scores["responder"]]
print("MLP better than RF on", round((resp["r_mlp"] > resp["r_rf"]).mean() * 100), "% of responders")

r_mean    0.4229
r_rf      0.5439
r_mlp     0.2308
r_mlp2    0.4361
dtype: float64
mse_mean    0.0105
mse_rf      0.0074
mse_mlp     0.0127
mse_mlp2    0.0078
dtype: float64
Responders only:
r_mean     0.4229
r_ridge    0.4985
r_rf       0.5439
r_xgb      0.4825
r_mlp      0.2308
dtype: float64
Responders MSE:
mse_mean    0.0105
mse_rf      0.0074
mse_mlp     0.0127
dtype: float64
MLP better than RF on 7 % of responders


In [26]:
scores["r_mlp2"] = row_r(P_mlp2, Yv)
scores["mse_mlp2"] = ((P_mlp2 - Yv) ** 2).mean(axis=1)

resp = scores[scores["responder"]]
print(resp[["r_mean", "r_rf", "r_mlp", "r_mlp2"]].mean().round(4))
print(resp[["mse_mean", "mse_rf", "mse_mlp", "mse_mlp2"]].mean().round(4))

r_mean    0.4229
r_rf      0.5439
r_mlp     0.2308
r_mlp2    0.4361
dtype: float64
mse_mean    0.0105
mse_rf      0.0074
mse_mlp     0.0127
mse_mlp2    0.0078
dtype: float64


In [27]:
scores.to_csv("../results/tables/06_model_scores.csv")
np.save("../results/models/P_mlp2.npy", P_mlp2)
print(scores.columns.tolist())

['r_mean', 'r_knn', 'mse_mean', 'mse_knn', 'responder', 'r_ridge', 'mse_ridge', 'r_rf', 'mse_rf', 'r_ridge_pca', 'r_xgb', 'mse_xgb', 'r_mlp', 'mse_mlp', 'r_mlp2', 'mse_mlp2']
